Based on [olivier's script](https://www.kaggle.com/code/ogrellier/xgb-classifier-upsampling-lb-0-283)

In [1]:
MAX_ROUNDS = 400
OPTIMIZE_ROUNDS = False
LEARNING_RATE = 0.07
EARLY_STOPPING_ROUNDS = 50

I recommend initially setting MAX_ROUNDS fairly high and using OPTIMIZE_ROUNDS to get an idea of the appropriate number of rounds (which, in my judgment, should be close to the maximum value of best_ntree_limit among all folds, maybe even a bit higher if your model is adequately regularized... or alternatively, you could set verbose=True and look at the details to try to find a number of rounds that works well for all folds). Then I would turn off OPTIMIZE_ROUNDS and set MAX_ROUNDS to the appropriate number of total rounds.

The problem with "early stopping" by choosing the best round for each fold is that it overfits to the validation data. It's therefore liable not to produce the optimal model for predicting test data, and if it's used to produce validation data for stacking/ensembling with other models, it would cause this one to have too much weight in the ensemble. Another possibility (and the default for XGBoost, it seems) is to use the round where the early stop actually happens (with the lag that verifies lack of improvement) rather than the best round. That solves the overfitting problem (provided the lag is long enough), but so far it doesn't seem to have helped. (I got a worse validation score with 20-round early stopping per fold than with a constant number of rounds for all folds, so the early stopping actually seemed to underfit.)

In [2]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from numba import jit
import time
import gc

In [3]:
@jit
def eval_gini(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_true = y_true[np.argsort(y_prob)]
    ntrue = 0
    gini = 0
    delta = 0
    n = len(y_true)
    for i in range(n-1, -1, -1):
        y_i = y_true[i]
        ntrue += y_i
        gini += y_i * delta
        delta += 1 - y_i
    gini = 1 - 2 * gini / (ntrue * (n - ntrue))
    return gini

In [4]:
def gini_xgb(preds, dtrain):
    labels = dtrain.get_label()
    gini_score = -eval_gini(labels, preds)
    return [('gini', gini_score)]

def add_noise(series, noise_level):
    return series * (1 + noise_level * np.random.randn(len(series)))

def target_encode(trn_series=None,
                  val_series=None,
                  tst_series=None,
                  target=None,
                  min_samples_leaf=1,
                  smoothing=1,
                  noise_level=0):
    assert len(trn_series) == len(target)
    assert trn_series.name == tst_series.name
    temp = pd.concat([trn_series, target], axis=1)

    averages = temp.groupby(by=trn_series.name)[target.name].agg(['mean', 'count'])
    smoothing = 1 / (1 + np.exp(-(averages['count'] - min_samples_leaf) / smoothing))

    prior = target.mean()

    averages[target.name] = prior * (1 - smoothing) + averages['mean'] * smoothing
    averages.drop(['mean', 'count'], axis=1, inplace=True)

    ft_trn_series = pd.merge(
        trn_series.to_frame(trn_series.name),
        averages.reset_index().rename(columns={'index': target.name, target.name: 'average'}),
        on=trn_series.name,
        how='left'
    )['average'].rename(trn_series.name + '_mean').fillna(prior)
    
    ft_trn_series.index = trn_series.index
    ft_val_series = pd.merge(
        val_series.to_frame(val_series.name),
        averages.reset_index().rename(columns={'index': target.name, target.name: 'average'}),
        on=val_series.name,
        how='left'
    )['average'].rename(trn_series.name + '_mean').fillna(prior)

    ft_val_series.index = val_series.index
    ft_tst_series = pd.merge(
        tst_series.to_frame(tst_series.name),
        averages.reset_index().rename(columns={'index': target.name, target.name: 'average'}),
        on=tst_series.name,
        how='left'
    )['average'].rename(trn_series.name + '_mean').fillna(prior)

    ft_tst_series.index = tst_series.index

    return add_noise(ft_trn_series, noise_level), add_noise(ft_val_series, noise_level), add_noise(ft_tst_series, noise_level)

In [5]:
train_df = pd.read_csv('./input/train.csv', na_values='-1')
test_df = pd.read_csv('./input/test.csv', na_values='-1')

In [6]:
train_features = [
    'ps_car_13',
    'ps_reg_03',
    'ps_ind_05_cat',
    'ps_ind_03',
    'ps_ind_15',
    'ps_reg_02',
    'ps_car_14',
    'ps_car_12',
    'ps_car_01_cat',
    'ps_car_07_cat',
    'ps_ind_17_bin',
    'ps_car_03_cat',
    'ps_reg_01',
    'ps_car_15',
    'ps_ind_01',
    'ps_ind_16_bin',
    'ps_ind_07_bin',
    'ps_car_06_cat',
    'ps_car_04_cat',
    'ps_ind_06_bin',
    'ps_car_09_cat',
    'ps_car_02_cat',
    'ps_ind_02_cat',
    'ps_car_11',
    'ps_car_05_cat',
    'ps_calc_09',
    'ps_calc_05',
    'ps_ind_08_bin',
    'ps_car_08_cat',
    'ps_ind_09_bin',
    'ps_ind_04_cat',
    'ps_ind_18_bin',
    'ps_ind_12_bin',
    'ps_ind_14'
]

combs = [
    ('ps_reg_01', 'ps_car_02_cat'),
    ('ps_reg_01', 'ps_car_04_cat')
]

In [7]:
id_test = test_df['id'].values
id_train = train_df['id'].values
y = train_df['target']

start = time.time()
for n_c, (f1, f2) in enumerate(combs):
    name1 = f1 + '_plus_' + f2
    print('current feature %60s %4d in %5.1f' % (name1, n_c + 1, (time.time() - start) / 60), end='')
    train_df[name1] = train_df[f1].apply(lambda x: str(x)) + '_' + train_df[f2].apply(lambda x: str(x))
    test_df[name1] = test_df[f1].apply(lambda x: str(x)) + '_' + test_df[f2].apply(lambda x: str(x))

    lbl = LabelEncoder()
    lbl.fit(list(train_df[name1].values) + list(test_df[name1].values))
    train_df[name1] = lbl.transform(list(train_df[name1].values))
    test_df[name1] = lbl.transform(list(test_df[name1].values))

    train_features.append(name1)

X = train_df[train_features]
test_df = test_df[train_features]

f_cats = [f for f in X.columns if "_cat" in f]

current feature                                 ps_reg_01_plus_ps_car_02_cat    1 in   0.0current feature                                 ps_reg_01_plus_ps_car_04_cat    2 in   0.0

In [8]:
y_valid_pred = 0*y
y_test_pred = 0

In [9]:
K = 5
kf = KFold(n_splits=K, random_state=1, shuffle=True)
np.random.seed(0)

In [10]:
model = XGBClassifier(
    n_estimators=MAX_ROUNDS,
    max_depth=4,
    objective='binary:logistic',
    learning_rate=LEARNING_RATE,
    subsample=.8,
    min_child_weight=6,
    colsample_bytree=.8,
    scale_pos_weight=1.6,
    gamma=10,
    reg_alpha=8,
    reg_lambda=1.3
)

In [12]:
for i, (train_index, test_index) in enumerate(kf.split(train_df)):
    y_train, y_valid = y.iloc[train_index].copy(), y.iloc[test_index]
    X_train, X_valid = X.iloc[train_index, :].copy(), X.iloc[test_index, :].copy()
    X_test = test_df.copy()
    print('\nFold ', i)

    for f in f_cats:
        X_train[f + '_avg'], X_valid[f + '_avg'], X_test[f + '_avg'] = target_encode(
            trn_series=X_train[f],
            val_series=X_valid[f],
            tst_series=X_test[f],
            target=y_train,
            min_samples_leaf=200,
            smoothing=10,
            noise_level=0
        )

    if OPTIMIZE_ROUNDS:
        eval_set = [(X_valid, y_valid)]
        fit_model = model.fit(X_train, y_train,
                                eval_set=eval_set,
                                eval_metric=gini_xgb,
                                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                                verbose=False)
        print('  Best N trees = ', model.best_n_tree_limit)
        print('  Best gini = ', model.best_score)
    else:
        fit_model = model.fit(X_train, y_train)
    
    pred = fit_model.predict_proba(X_valid)[:,1]
    print('  Gini = ', eval_gini(y_valid.values, pred))
    y_valid_pred.iloc[test_index] = pred

    y_test_pred += fit_model.predict_proba(X_test)[:, 1]

    del X_test, X_train, X_valid, y_train
    
y_test_pred /= K

print('\nGini for full training set:')
eval_gini(y.values, y_valid_pred.values)


Fold  0
  Gini =  0.2826992327122274


C:\Users\USER\AppData\Local\Temp\ipykernel_14004\3666721472.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.02561709 0.04883473 0.07381289 ... 0.04027547 0.03847545 0.0504147 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  y_valid_pred.iloc[test_index] = pred



Fold  1
  Gini =  0.277224351046245

Fold  2
  Gini =  0.27179232815004395

Fold  3
  Gini =  0.2969724066053364

Fold  4
  Gini =  0.28029050894463814

Gini for full training set:


0.2815202003723114

In [13]:
val = pd.DataFrame()
val['id'] = id_train
val['target'] = y_valid_pred.values
val.to_csv('xgb_valid.csv', float_format='%.6f', index=False)

In [15]:
sub = pd.DataFrame()
sub['id'] = id_test
sub['target'] = y_test_pred
sub.to_csv('xgb_submit.csv', float_format='%.6f', index=False)

Noetes:

version 16. Baseline best CV=.2832, LB=.282

version 15. Ntree optimization for baseline

version 21. Verbose version of baseline optimization

version 22. Baseline + per-fold early stopping after 20 rounds

version 23. Back to baseline.

version 24. Some parameter tuning.

version 25. Re-published to make it visible.

version 26. A little more tuning.

version 27. More tuning, get rid of upsampling (using **scale_pos_weight** instead), Set OPTIMIZE_ROUNDS and verbose temporarily

version 28. MAX_ROUNDS=300 as a compromise

version 29. Substantively identical. (Turn off now-irrelevant verbose.)

version 30: Still substantively identical. Some visual cleanup.

version 35. More tuning. CV went up but LB sorts lower (still .283)

version 36. Identical (except turn off irrelevant verbose). Republished to make it visible.

version 37-42. More tuning (gamma=10, alpha=8). LB .284 (*end zone dance*).

version 43. More tuning (min_child_weight=6). LB score has considerably improved according to sort, but still .284